In [ ]:
import pandas as pd
import re
import matplotlib.pyplot as plt

In [ ]:
def normalize_implementation(impl: str) -> str:
    impl = impl.lower()
    if "default implementation" in impl:
        return "default"
    elif "hashmap implementation" in impl:
        if "no wildcard support" in impl:
            return "hashmap_no_wildcard"
        elif "overtaking" in impl:
            return "hashmap_overtaking_wildcard"
        else:
            return "hashmap_with_wildcard"
    return impl.strip()

def parse_log_to_df(log_file_path):
    rows = []
    num_threads = None
    wildcard_percentage = 0.0
    sequence_id=0

    with open(log_file_path, "r") as file:
        lines = file.readlines()


    i = 0
    while i < len(lines):
        line = lines[i].strip()

        # get operation sequence
        if line == "Generate Operation Sequence":
            sequence_id+=1


        # Capture number of threads
        match_threads = re.match(r"Run with (\d+) Threads", line)
        if match_threads:
            num_threads = int(match_threads.group(1))
            i += 1
            continue

        # Capture wildcard percentage
        match_wildcard_pct = re.match(r"Percentage of Operations with Wildcards:\s+([\d.]+)\s*%", line)
        if match_wildcard_pct:
            wildcard_percentage = float(match_wildcard_pct.group(1))
            i += 1
            continue

        # Capture implementation line
        if "Implementation" in line:
            implementation_line = line
            implementation = normalize_implementation(implementation_line)
            wildcard_usage = "with wildcards" in implementation_line.lower()
            if not wildcard_usage:
                wildcard_percentage=0.0


            # Next line: operations/time/ops_per_sec
            ops_line = lines[i+1].strip()
            ops_match = re.match(
                r"Number of Operations:\s+(\d+)\s+in\s+([\d.]+)\s+ms\s+\(([\d.]+)\s+ops/sec\)",
                ops_line
            )
            if ops_match:
                num_ops = int(ops_match.group(1))
                time = float(ops_match.group(2))
                ops_per_sec = float(ops_match.group(3))
            else:
                num_ops, time, ops_per_sec = None, None, None

            # PRQ line
            prq_line = lines[i+2].strip()
            prq_match = re.search(r"PRQ max size:\s*(\d+)", prq_line)
            PRQ_max = int(prq_match.group(1)) if prq_match else None

            # UMQ line
            umq_line = lines[i+3].strip()
            umq_match = re.search(r"UMQ max size:\s*(\d+)", umq_line)
            UMQ_max = int(umq_match.group(1)) if umq_match else None

            rows.append({
                "sequence_id":sequence_id,
                "num_threads": num_threads,
                "implementation": implementation,
                "wildcard_usage": wildcard_usage,
                "wildcard_percentage": wildcard_percentage,
                "num_ops": num_ops,
                "time": time,
                "ops_per_sec": ops_per_sec,
                "PRQ_max": PRQ_max,
                "UMQ_max": UMQ_max
            })

            i += 4
            continue

        i += 1

    return pd.DataFrame(rows)


df = parse_log_to_df("sample.log")
df

In [ ]:
# Compute max(PRQ_max, UMQ_max)
df["max_queue"] = df[["PRQ_max", "UMQ_max"]].max(axis=1)

# Separate data by wildcard_usage
df_no_wildcard = df[df["wildcard_usage"] == False]
df_with_wildcard = df[df["wildcard_usage"] == True]

# Plot without wildcard usage
plt.figure()
for impl, group in df_no_wildcard.groupby("implementation"):
    plt.scatter(group["max_queue"], group["ops_per_sec"], label=impl)
plt.ylabel("Operations per Second")
plt.xlabel("Maximum Queue size max(PRQ,UMQ)")
plt.title("Performance vs Queue Size (No Wildcards)")
plt.legend()
plt.grid(True)
plt.show()

# Plot with wildcard usage
plt.figure()
for impl, group in df_with_wildcard.groupby("implementation"):
    plt.scatter(group["max_queue"], group["ops_per_sec"], label=impl)
plt.ylabel("Operations per Second")
plt.xlabel("Maximum Queue size max(PRQ,UMQ)")
plt.title("Performance vs Queue Size (With Wildcards)")
plt.legend()
plt.grid(True)
plt.show()
